# Week 8, Day 4: A bargain-hunting agent loop

**Lesson:** [week8/day4.ipynb](../../../week8/day4.ipynb), *Autonomous planner agent and the agent framework*. **About 40 minutes.**

**What you practice here**

- Describing tools to the model in JSON: a name, a description, and parameters as a JSON schema, in the `tools` list.
- Running the tool calls the model asks for: reading `message.tool_calls`, decoding each call's arguments, and answering each call with a `"tool"` message that carries its id.
- The agent loop: calling the model with its tools again while it asks for them, until it answers in text.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import os, pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT is None and ("google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ):
    # in Colab: fetch the checker from your fork on GitHub, and the packages Colab lacks
    import subprocess, urllib.request
    ROOT = pathlib.Path("/content/fork")
    (ROOT / "plan").mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/starlest/llm_engineering/main/plan/check.py", ROOT / "plan" / "check.py")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "python-dotenv", "tiktoken"], check=False)
    print("In Colab, the checks run here too. To keep your work, use File > Save a copy in GitHub, to this notebook's path in your fork.")
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week8/day4", globals())
else:  # outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here.")

## The task

In the lesson, `gpt-5.1` runs the whole deal hunt itself. It gets three tools, described in JSON, and a plan in its
messages. Each time it asks for tools, `handle_tool_call` runs them, and the loop sends the results back, until the
model answers in text. Build that loop for a hunt for wrestling merchandise: the model searches a marketplace by a
keyword that it chooses itself (unlike the lesson's scan, which takes no argument), estimates what each find is worth,
and alerts the user once about the best bargain.

`hunt` takes the client as its first parameter and makes the Chat Completions calls itself, and `handle_tool_calls`
reads the model's reply as the SDK gives it. The checks hand them a stand-in shaped like the real client: its replies
ask for your tools the way the real API does, with the arguments as JSON text, and it refuses the messages that the
real API would refuse. The tools come in as a dict of Python functions, so that the checks can pass pretend ones that
record what they were asked.

## What each call should give

```python
[tool["function"]["name"] for tool in TOOLS]
# ['search_listings', 'estimate_value', 'alert_user']

handle_tool_calls(message, FUNCTIONS)     # message: a reply of the model that asks for estimate_value
# [{'role': 'tool', 'content': '"The estimated true value of ... is $90.00"', 'tool_call_id': 'call_1'}]

hunt(client, FUNCTIONS, "Find me a bargain on Undertaker merchandise")
# asks the model with TOOLS, runs each tool it asks for, asks again, and so on, then returns its answer, such as
# 'I alerted you to the 1991 Hasbro Undertaker figure: $45.50, and worth about $90.'
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.
import json

### 1 · `TOOLS`

Describes the three tools to the model: the list that you pass as the call's tools, in the lesson's JSON. Each entry
is a function tool with a name, a description and parameters, for these tools:

- `search_listings`, with one parameter: `keyword`, a string;
- `estimate_value`, with `description`, a string;
- `alert_user`, with `description` (a string), `price` (a number), `estimated_value` (a number) and `url` (a string).

Give each tool a description of at least 10 characters that tells the model when to use it; say that `alert_user` is
to be called once only. Give each tool's parameters the lesson's shape: a JSON schema of type object whose properties
each have a type and a description, with all of them required and no other properties allowed.

<details><summary>Hint</summary>

One of the three, in the lesson's shape:

```python
{"type": "function", "function": {
    "name": "estimate_value",
    "description": "...",
    "parameters": {"type": "object",
                   "properties": {"description": {"type": "string", "description": "..."}},
                   "required": ["description"],
                   "additionalProperties": False}}}
```

</details>

In [ ]:
TOOLS = []

In [ ]:
check("TOOLS")

### 2 · `handle_tool_calls(message, functions)`

Runs each tool that the model asked for, and returns one tool message per call. `message` is the model's reply as the
SDK gives it, and `functions` maps each tool's name to the Python function that does it, such as
`{"search_listings": search_listings, ...}`. For each tool call in the message, in order: find its function by the
call's name, and call it with the call's arguments, which arrive as JSON text, as keyword arguments. Answer it with a
message whose role is `"tool"`, whose content is the result written as JSON text, and which carries the call's id, as
the lesson does. A name that is not in `functions` still gets an answer, the JSON text of `{"error": ...}` with a text
that names the tool, since the API refuses the next request if any call is left unanswered. Return the list.

<details><summary>Hint</summary>

```python
for call in message.tool_calls:
    arguments = json.loads(call.function.arguments)
    ... functions[call.function.name](**arguments) ...
    {"role": "tool", "content": json.dumps(result), "tool_call_id": call.id}
```

</details>

In [ ]:
def handle_tool_calls(message, functions):
    # Write handle_tool_calls here, as described above.
    raise NotImplementedError("handle_tool_calls is not written yet")

In [ ]:
check("handle_tool_calls")

### 3 · `hunt(client, functions, request, model="gpt-5.1")`

Runs the agent loop: the model calls tools until it has hunted down a bargain, then answers in text. Start the
messages with a system message of your own, at least 20 characters long, that gives the agent its job and its plan
(search the marketplace, estimate each find, alert the user once about the best bargain, then answer in one
sentence), and a user message holding `request`. Make a Chat Completions call with the model `model`, the messages and
`TOOLS`. While the reply's finish reason is `"tool_calls"`, append the reply's message as it is, then the answers from
`handle_tool_calls`, and call again with the same tools. Return the text of the last reply.

<details><summary>Hint</summary>

```python
response = client.chat.completions.create(model=model, messages=messages, tools=TOOLS)
while response.choices[0].finish_reason == "tool_calls":
    message = response.choices[0].message
    ...
return response.choices[0].message.content
```

</details>

In [ ]:
def hunt(client, functions, request, model="gpt-5.1"):
    # Write hunt here, as described above.
    raise NotImplementedError("hunt is not written yet")

In [ ]:
check("hunt")

## Check everything

The cell below runs all 7 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 8, Day 4 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. Write the steps in the cell at the end; they may call a model with your key, and the checks leave them out.

A hunt with `gpt-5.1` takes four or five requests and costs about a cent; the pretend tools cost nothing. Step 5's
real agent makes over a dozen model calls and wakes the GPU container on Modal.

1. Load your keys with `load_dotenv(override=True)`, and make `client = OpenAI()`.
2. Write three pretend tools for the hunt, each printing what it was asked, as the lesson's pretend functions do:
   `search_listings(keyword)` returns the `MARKET` listings in the cell below whose title holds the keyword;
   `estimate_value(description)` returns a made-up value as text, as the lesson's always says $300; and
   `alert_user(description, price, estimated_value, url)` returns `"notification sent ok"`. Map their names to them in
   a dict, `FUNCTIONS`.
3. Print `hunt(client, FUNCTIONS, "Find me a bargain on Undertaker merchandise")`. Watch which tools the model calls,
   and in what order.
4. Ask for something the market lacks, such as Ultimate Warrior merchandise, and see how the agent copes.
5. Optional: watch the course's agent do the same with the real scanner, ensemble and messenger, as the lesson's last
   cells do: `AutonomousPlanningAgent(collection).plan()`, run from the course's `week8` folder
   (`os.chdir(ROOT / "week8")`). It needs the pricer deployed on Modal, the lesson's `products_vectorstore`,
   `deep_neural_network.pth`, Ollama running (or `PRICER_PREPROCESSOR_MODEL` set to a Groq model), your Pushover keys
   and an Anthropic key.

<details><summary>Stuck? One way to write it</summary>

```python
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
client = OpenAI()

MARKET = [
    {"title": "Undertaker 1991 Hasbro figure with urn", "price": 45.5, "url": "https://merch.example/l/101"},
    {"title": "Undertaker Elite figure, Deadman era", "price": 28.0, "url": "https://merch.example/l/107"},
    {"title": "Winged Eagle replica title belt", "price": 349.0, "url": "https://merch.example/l/102"},
    {"title": "WrestleMania III program, Pontiac Silverdome", "price": 60.0, "url": "https://merch.example/l/103"},
    {"title": "Bret Hart Hasbro figure, 1992", "price": 38.0, "url": "https://merch.example/l/106"},
]


def search_listings(keyword):
    """Pretends to search the marketplace: the listings whose title mentions the keyword."""
    print(f"Pretend search for {keyword!r}")
    return [item for item in MARKET if keyword.lower() in item["title"].lower()]  # handle_tool_calls writes the JSON


def estimate_value(description):
    """Pretends to estimate what an item is worth, as the lesson's pretend function does."""
    print(f"Pretend estimate for {description[:40]}...")
    return f"The estimated true value of {description} is ${len(description) * 1.5:.2f}"


def alert_user(description, price, estimated_value, url):
    """Pretends to alert the user."""
    print(f"Pretend alert: {description} for ${price}, worth about ${estimated_value}: {url}")
    return "notification sent ok"


FUNCTIONS = {"search_listings": search_listings, "estimate_value": estimate_value, "alert_user": alert_user}
print(hunt(client, FUNCTIONS, "Find me a bargain on Undertaker merchandise"))
```

```python
print(hunt(client, FUNCTIONS, "Find me a bargain on Ultimate Warrior merchandise"))
```

```python
import os
import sys

import chromadb

os.chdir(ROOT / "week8")  # the course's agents find their files from its week8 folder
sys.path.insert(0, str(ROOT / "week8"))
from agents.autonomous_planning_agent import AutonomousPlanningAgent

collection = chromadb.PersistentClient(path="products_vectorstore").get_collection("products")  # raises if missing
print(AutonomousPlanningAgent(collection).plan())
```

</details>

In [ ]:
# Try it for real: write the steps above here, in as many cells as you like. This cell may call a model with your key; the checks leave it out.
# The listings on the marketplace your pretend search reads:
MARKET = [
    {"title": "Undertaker 1991 Hasbro figure with urn", "price": 45.5, "url": "https://merch.example/l/101"},
    {"title": "Undertaker Elite figure, Deadman era", "price": 28.0, "url": "https://merch.example/l/107"},
    {"title": "Winged Eagle replica title belt", "price": 349.0, "url": "https://merch.example/l/102"},
    {"title": "WrestleMania III program, Pontiac Silverdome", "price": 60.0, "url": "https://merch.example/l/103"},
    {"title": "Bret Hart Hasbro figure, 1992", "price": 38.0, "url": "https://merch.example/l/106"},
]